# 111 — Contextual RL Token reconstruction and Q10 ablation

This implements the **representation-learning stage** of [RL Token](https://arxiv.org/html/2604.23073v2): extract the frozen VLM's **final contextualized prefix states**, train an encoder readout token to autoregressively reconstruct them, freeze it, then train a root-MC Q10 on the resulting single state token. It is distinct from the learned readout token already inside `QPlanningCritic`.

Local adaptation: final states are pooled to 128 tokens; the VLA is frozen and there is no separate RL actor. The source artifacts are rich enough for final-layer extraction; compact v4 branch artifacts are not, so this ablation currently supports **root MC only**, not tree TD. Start with `EXTRACT_LIMIT=1` to verify the pinned LeRobot API and shapes. Full feature extraction and training are opt-in.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
EXTRAS = 'sim'
SETUP_ENV = False
REPO_REF = 'codex/smolvla-q-program'
import urllib.request
exec(urllib.request.urlopen('https://raw.githubusercontent.com/ArjunS07/cs159-sp26/codex/smolvla-q-program/pnp-vla/scripts/colab_bootstrap.py').read().decode())


In [ ]:
from pathlib import Path
import torch
from pnp.smolvla_q_selection import FRESH8_EXPERIMENT, FRESH8_KINDS, fresh8_snapshot_key
from pnp.smolvla_rl_token import (
    prepare_contextual_features, train_rl_token, materialize_root_tokens)
from pnp.smolvla_tree_bellman_finetune import load_or_create_tree_snapshot, prepare_tree_bellman_cache
from pnp.smolvla_tree_collection import _source_rows
from pnp.store import SupabaseStore

TREE_LIMIT = 800
FEATURE_ROOT = Path('/content/drive/MyDrive/pnp_smolvla_rlt/contextual_features')
RLT_CHECKPOINT = Path('/content/drive/MyDrive/pnp_smolvla_rlt/rlt_reconstruction.pt')
ROOT_TOKEN_DIR = Path('/content/drive/MyDrive/pnp_smolvla_rlt/root_tokens')
Q_OUTPUT_ROOT = Path('/content/drive/MyDrive/pnp_smolvla_q10_fresh8')
CACHE_ROOT = Path('/content/smolvla_q10_fresh8_cache')
EXTRACT_LIMIT = 1  # smoke; set to None for all 800 source rollouts
RUN_EXTRACTION = False
RUN_RLT_TRAIN = False
RUN_RLT_Q_TRAIN = False
assert torch.cuda.is_available(), 'Use a GPU runtime for VLM feature extraction.'
store = SupabaseStore()
snapshot = load_or_create_tree_snapshot(
    store=store, snapshot_key=fresh8_snapshot_key(TREE_LIMIT), tree_limit=TREE_LIMIT,
    experiment=FRESH8_EXPERIMENT, candidate_kinds=FRESH8_KINDS)
cache = prepare_tree_bellman_cache(
    snapshot=snapshot, cache_root=CACHE_ROOT, gamma=1.0,
    success_reward=True, store=store)
source_rows = _source_rows(store)
source_by_path = {str(r['training_data_path']): r['rollout_id'] for r in source_rows}
groups = {g['candidate_group_id']: g for g in snapshot['groups']}
def source_ids(group_ids):
    return {source_by_path[groups[g]['candidates'][0]['training_data_path']] for g in group_ids}
train_ids = source_ids(cache['train_group_ids'])
validation_ids = source_ids(cache['validation_group_ids'])
assert not train_ids & validation_ids
assert len(train_ids | validation_ids) == TREE_LIMIT
print({'train_source_episodes': len(train_ids), 'validation_source_episodes': len(validation_ids)})


In [ ]:
if RUN_EXTRACTION:
    print(prepare_contextual_features(
        output_root=FEATURE_ROOT, train_rollout_ids=train_ids,
        validation_rollout_ids=validation_ids, limit=EXTRACT_LIMIT,
        store=store))
else:
    print('No extraction. Set RUN_EXTRACTION=True; keep EXTRACT_LIMIT=1 for first smoke.')


In [ ]:
if RUN_RLT_TRAIN:
    assert len(list((FEATURE_ROOT/'train').glob('*.npz'))) == len(train_ids)
    assert len(list((FEATURE_ROOT/'validation').glob('*.npz'))) == len(validation_ids)
    print(train_rl_token(feature_root=FEATURE_ROOT, output_path=RLT_CHECKPOINT,
                         epochs=20, batch_size=16, device='cuda'))
else:
    print('RLT reconstruction not started.')


In [ ]:
if RUN_RLT_Q_TRAIN:
    assert RLT_CHECKPOINT.is_file()
    feature_manifest = materialize_root_tokens(
        snapshot=snapshot, cache=cache, source_rows=source_rows,
        checkpoint=RLT_CHECKPOINT, feature_root=FEATURE_ROOT,
        output_root=ROOT_TOKEN_DIR, device='cuda')
    print(feature_manifest)
    from pnp.smolvla_success_critic import train_smolvla_success_q10
    report = train_smolvla_success_q10(
        arm='root_mc', output_root=Q_OUTPUT_ROOT, cache_root=CACHE_ROOT,
        tree_limit=TREE_LIMIT, snapshot_key=fresh8_snapshot_key(TREE_LIMIT),
        experiment=FRESH8_EXPERIMENT, candidate_kinds=FRESH8_KINDS,
        root_token_dir=ROOT_TOKEN_DIR, updates=2000, resume=True,
        device='cuda', store=store)
    print(report)
else:
    print('RL Token Q ablation not started.')
